# 01 Data Cleaning

This notebook loads the raw e-commerce dataset, checks quality issues, converts date and numeric fields, creates analysis-ready time columns, and saves the cleaned dataset for the next notebooks.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd

from cleaning import clean_ecommerce_data, missing_value_summary
from utils import load_data, save_dataframe

## Load Raw Data

In [2]:
df_raw = load_data("ecommerce_data.csv")
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

Raw dataset shape: (1500, 10)


,Order ID,Order Date,Customer ID,Product Name,Category,Sales,Quantity,Discount,Profit,Region
0,ORD-20230001,2023-01-31,CUST-1047,Air Fryer,Home & Kitchen,85.42,1,0.00,22.89,West
1,ORD-20230002,2023-01-07,CUST-1388,Wireless Mouse,Electronics,99.86,3,0.15,14.98,East
2,ORD-20230003,2023-02-14,CUST-1373,Lipstick Pack,Beauty,11.69,1,0.25,-3.64,South
3,ORD-20230004,2023-07-16,CUST-1360,Denim Jacket,Fashion,79.09,1,0.10,20.73,Central
4,ORD-20230005,2024-01-15,CUST-1181,Bluetooth Speaker,Electronics,660.96,6,0.20,82.60,West


In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Order ID      1500 non-null   str    
 1   Order Date    1500 non-null   str    
 2   Customer ID   1500 non-null   str    
 3   Product Name  1500 non-null   str    
 4   Category      1500 non-null   str    
 5   Sales         1500 non-null   float64
 6   Quantity      1500 non-null   int64  
 7   Discount      1500 non-null   float64
 8   Profit        1500 non-null   float64
 9   Region        1500 non-null   str    
dtypes: float64(3), int64(1), str(6)
memory usage: 117.3 KB


## Initial Quality Checks

In [4]:
missing_value_summary(df_raw)

,missing_count,missing_percent
Order ID,0,0.0
Order Date,0,0.0
Customer ID,0,0.0
Product Name,0,0.0
Category,0,0.0
Sales,0,0.0
Quantity,0,0.0
Discount,0,0.0
Profit,0,0.0
Region,0,0.0


In [5]:
duplicate_count = df_raw.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


## Clean Data

In [6]:
df_clean = clean_ecommerce_data(df_raw)
print(f"Clean dataset shape: {df_clean.shape}")
df_clean.head()

Clean dataset shape: (1500, 14)


,Order ID,Order Date,Customer ID,Product Name,Category,Sales,Quantity,Discount,Profit,Region,Year,Month,Month Name,Profit Margin
0,ORD-20230001,2023-01-31,CUST-1047,Air Fryer,Home & Kitchen,85.42,1,0.00,22.89,West,2023,1,January,0.267970
1,ORD-20230002,2023-01-07,CUST-1388,Wireless Mouse,Electronics,99.86,3,0.15,14.98,East,2023,1,January,0.150010
2,ORD-20230003,2023-02-14,CUST-1373,Lipstick Pack,Beauty,11.69,1,0.25,-3.64,South,2023,2,February,-0.311377
3,ORD-20230004,2023-07-16,CUST-1360,Denim Jacket,Fashion,79.09,1,0.10,20.73,Central,2023,7,July,0.262106
4,ORD-20230005,2024-01-15,CUST-1181,Bluetooth Speaker,Electronics,660.96,6,0.20,82.60,West,2024,1,January,0.124970


In [7]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Order ID       1500 non-null   str           
 1   Order Date     1500 non-null   datetime64[us]
 2   Customer ID    1500 non-null   str           
 3   Product Name   1500 non-null   str           
 4   Category       1500 non-null   str           
 5   Sales          1500 non-null   float64       
 6   Quantity       1500 non-null   int64         
 7   Discount       1500 non-null   float64       
 8   Profit         1500 non-null   float64       
 9   Region         1500 non-null   str           
 10  Year           1500 non-null   int32         
 11  Month          1500 non-null   int32         
 12  Month Name     1500 non-null   str           
 13  Profit Margin  1500 non-null   float64       
dtypes: datetime64[us](1), float64(4), int32(2), int64(1), str(6)
memory usage: 152.5 KB

In [8]:
df_clean[["Order Date", "Year", "Month", "Month Name"]].head()

,Order Date,Year,Month,Month Name
0,2023-01-31,2023,1,January
1,2023-01-07,2023,1,January
2,2023-02-14,2023,2,February
3,2023-07-16,2023,7,July
4,2024-01-15,2024,1,January


In [9]:
numeric_summary = df_clean[["Sales", "Quantity", "Discount", "Profit", "Profit Margin"]].describe().T
numeric_summary

,count,mean,std,min,25%,50%,75%,max
Sales,1500.0,150.774473,164.525686,7.690000,47.752500,98.580000,188.4875,1590.450000
Quantity,1500.0,2.218000,1.335040,1.000000,1.000000,2.000000,3.0000,6.000000
Discount,1500.0,0.110900,0.087947,0.000000,0.050000,0.100000,0.2000,0.300000
Profit,1500.0,31.170927,36.124765,-16.550000,9.777500,20.030000,38.7750,417.100000
Profit Margin,1500.0,0.211860,0.129658,-1.370897,0.158421,0.224523,0.2844,0.531877


## Save Cleaned Data

In [10]:
save_path = save_dataframe(df_clean, "ecommerce_data_cleaned.csv")
save_path

WindowsPath('C:/Users/asus/OneDrive/Documents/ecommerce-business-analysis/data/cleaned/ecommerce_data_cleaned.csv')

The cleaned dataset is now ready for exploratory analysis in the next notebook.